In [1]:
import numpy as np
import torch
from torch import nn
from torchvision import datasets, transforms

print(torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

False


In [ ]:
from torch.utils.data import Subset, DataLoader
from sqrlearning.server import FedAvgServer, KrumServer, TrimmedMeanServer
from sqrlearning.client import Client, BackdoorClient, NeurotoxinClient, A3FLClient, PonderClient, DPClient
from sqrlearning.client.BackdoorClient import evaluate_asr
from models.nets import MnistCNN
from dataset.noniid import mnist_data
from dataset.trigger_fn import create_trigger_fn

num_channels = 1
num_classes = 10
local_epochs = 5
num_clients = 5
batch_size = 64
num_attackers = 1

trigger_fn=create_trigger_fn()
target_label=0

dataset_train, dataset_test, dataset_clients = mnist_data(num_clients, alpha=0.9)
img_size = dataset_train[0][0].shape
print(img_size)

for dc in dataset_clients:
    print(len(dc))

iid setting in MNIST
torch.Size([1, 28, 28])
13802
10591
14874
8293
12440


In [3]:
def evaluate(net, dataset, device, batch_size=256, criterion=None):
    """
    返回 (accuracy, average_loss)，均为 Python float。
    不修改 net 的 training 状态——内部保存并恢复。
    """
    was_training = net.training
    net.eval()

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    total, correct, loss_sum = 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = net(images)
        loss_sum += criterion(logits, labels).item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    if was_training:
        net.train()          # 恢复调用前的模式，避免影响后续训练

    return correct / total, loss_sum / total

In [4]:
def train(server, clients, frac=1.0):
    global_weights = server.get_weights()

    num_clients = len(clients)
    m = max(int(frac * num_clients), 1)

    idxs_users = np.random.choice(range(num_clients), m, replace=False)
    selected = [clients[idx] for idx in idxs_users]
    print(idxs_users, "in this round")

    for client in selected:
        client.train(global_weights)
    
    server.aggregate(selected)


In [ ]:
def test_dpclient(rounds, batch_size, target_epsilon, target_delta=1.25e-4):
    
    server = FedAvgServer(MnistCNN(num_channels, num_classes), device=device)
        # server = TrimmedMeanServer(CNNMnist(num_channels, num_classes), num_attackers=1, device=device)

    clients = [DPClient(f"Client-{i}",
                    dataset_clients[i], MnistCNN,
                    net_para = dict(num_channels=num_channels, num_classes=num_classes),
                    batch_size=batch_size, num_epochs=local_epochs,
                    total_rounds=rounds, target_epsilon=target_epsilon, target_delta=target_delta,
                    device=device)
                    for i in range(num_clients)
               ]
    for c in clients:
        print(f"{c.name}: {c.data_size} samples, {c.noise_info}")

    for round in range(1, rounds + 1):
        train(server, clients)

        acc_train, loss_avg = evaluate(server.net, dataset_train, device)
        # acc_attacher, _ = evaluate(clients[0].net, clients[0].train_data, device)

        print(f"Round {round:2d}: Training acc: {100*acc_train:.2f}%, global loss {loss_avg:.3f}")
        # print(f"Round {round:2d}: Attacker training acc: {100*acc_attacher:.2f}%")
        # asr_global = evaluate_asr(server.net, dataset_test, trigger_fn, target_label, device)
        # asr_local = evaluate_asr(clients[0].net, dataset_test, trigger_fn, target_label, device)
        # print(f"Round {round:2d}: global ASR={100*asr_global:.2f}%, attacker-local ASR={100*asr_local:.2f}%")

    acc_test, _ = evaluate(server.net, dataset_test, device)
    print(f"Testing accuracy: {100*acc_test:.2f}%")

    for c in clients:
        c.cleanup()

In [6]:
test_dpclient(15, 128, 50, 1e-3)

C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


Client-0: 13802 samples, sigma=0.42629241943359375
Client-1: 10591 samples, sigma=0.43964385986328125
Client-2: 14874 samples, sigma=0.4224395751953125
Client-3: 8293 samples, sigma=0.4531097412109375
Client-4: 12440 samples, sigma=0.4310798645019531
[2 3 0 1 4] in this round


c:\Users\yuloeh\Projects\Fed-Backdoor\sqrlearning\client\DPClient.py:47: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Client-2: ε=10.736 (target=50), δ=0.001
Client-3: ε=10.476 (target=50), δ=0.001
Client-0: ε=10.672 (target=50), δ=0.001
Client-1: ε=10.571 (target=50), δ=0.001
Client-4: ε=10.636 (target=50), δ=0.001
Round  1: Training acc: 30.68%, global loss 2.233
[0 4 2 3 1] in this round
Client-0: ε=14.915 (target=50), δ=0.001
Client-4: ε=14.879 (target=50), δ=0.001
Client-2: ε=14.994 (target=50), δ=0.001
Client-3: ε=14.720 (target=50), δ=0.001
Client-1: ε=14.815 (target=50), δ=0.001
Round  2: Training acc: 40.55%, global loss 2.080
[1 3 2 0 4] in this round
Client-1: ε=18.396 (target=50), δ=0.001
Client-3: ε=18.306 (target=50), δ=0.001
Client-2: ε=18.584 (target=50), δ=0.001
Client-0: ε=18.491 (target=50), δ=0.001
Client-4: ε=18.458 (target=50), δ=0.001
Round  3: Training acc: 55.56%, global loss 1.861
[0 3 2 4 1] in this round
Client-0: ε=21.726 (target=50), δ=0.001
Client-3: ε=21.552 (target=50), δ=0.001
Client-2: ε=21.830 (target=50), δ=0.001
Client-4: ε=21.695 (target=50), δ=0.001
Client-1: ε=

In [7]:
test_dpclient(15, 256, 50, 1e-3)

C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the smallest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


Client-0: 13802 samples, sigma=0.4640960693359375
Client-1: 10591 samples, sigma=0.4802131652832031
Client-2: 14874 samples, sigma=0.4587554931640625
Client-3: 8293 samples, sigma=0.4971504211425781
Client-4: 12440 samples, sigma=0.47016143798828125
[1 4 0 2 3] in this round
Client-1: ε=10.298 (target=50), δ=0.001
Client-4: ε=10.361 (target=50), δ=0.001
Client-0: ε=10.401 (target=50), δ=0.001
Client-2: ε=10.437 (target=50), δ=0.001
Client-3: ε=10.196 (target=50), δ=0.001
Round  1: Training acc: 13.60%, global loss 2.286
[3 1 4 0 2] in this round
Client-3: ε=14.444 (target=50), δ=0.001
Client-1: ε=14.545 (target=50), δ=0.001
Client-4: ε=14.607 (target=50), δ=0.001
Client-0: ε=14.647 (target=50), δ=0.001
Client-2: ε=14.683 (target=50), δ=0.001
Round  2: Training acc: 25.88%, global loss 2.250
[4 2 1 0 3] in this round
Client-4: ε=18.198 (target=50), δ=0.001
Client-2: ε=18.270 (target=50), δ=0.001
Client-1: ε=18.138 (target=50), δ=0.001
Client-0: ε=18.236 (target=50), δ=0.001
Client-3: ε=

In [8]:
test_dpclient(15, 128, 100, 1e-3)

Client-0: 13802 samples, sigma=0.35805702209472656
Client-1: 10591 samples, sigma=0.36696434020996094
Client-2: 14874 samples, sigma=0.3554725646972656
Client-3: 8293 samples, sigma=0.375823974609375
Client-4: 12440 samples, sigma=0.3612709045410156
[4 1 0 3 2] in this round
Client-4: ε=19.197 (target=100), δ=0.001
Client-1: ε=19.119 (target=100), δ=0.001
Client-0: ε=19.241 (target=100), δ=0.001
Client-3: ε=19.000 (target=100), δ=0.001
Client-2: ε=19.323 (target=100), δ=0.001
Round  1: Training acc: 26.23%, global loss 2.232
[3 1 0 4 2] in this round
Client-3: ε=27.302 (target=100), δ=0.001
Client-1: ε=27.423 (target=100), δ=0.001
Client-0: ε=27.547 (target=100), δ=0.001
Client-4: ε=27.502 (target=100), δ=0.001
Client-2: ε=27.654 (target=100), δ=0.001
Round  2: Training acc: 37.96%, global loss 2.118
[0 2 1 4 3] in this round
Client-0: ε=34.672 (target=100), δ=0.001
Client-2: ε=34.799 (target=100), δ=0.001
Client-1: ε=34.551 (target=100), δ=0.001
Client-4: ε=34.628 (target=100), δ=0.00

In [9]:
test_dpclient(15, 256, 100, 1e-3)

Client-0: 13802 samples, sigma=0.38295745849609375
Client-1: 10591 samples, sigma=0.39325714111328125
Client-2: 14874 samples, sigma=0.3795051574707031
Client-3: 8293 samples, sigma=0.403900146484375
Client-4: 12440 samples, sigma=0.38684844970703125
[4 0 2 1 3] in this round
Client-4: ε=18.859 (target=100), δ=0.001
Client-0: ε=18.908 (target=100), δ=0.001
Client-2: ε=18.952 (target=100), δ=0.001
Client-1: ε=18.778 (target=100), δ=0.001
Client-3: ε=18.648 (target=100), δ=0.001
Round  1: Training acc: 13.41%, global loss 2.270
[1 3 2 0 4] in this round
Client-1: ε=27.077 (target=100), δ=0.001
Client-3: ε=26.945 (target=100), δ=0.001
Client-2: ε=27.253 (target=100), δ=0.001
Client-0: ε=27.208 (target=100), δ=0.001
Client-4: ε=27.158 (target=100), δ=0.001
Round  2: Training acc: 18.36%, global loss 2.233
[0 4 3 1 2] in this round
Client-0: ε=34.342 (target=100), δ=0.001
Client-4: ε=34.293 (target=100), δ=0.001
Client-3: ε=34.086 (target=100), δ=0.001
Client-1: ε=34.214 (target=100), δ=0.0

In [10]:
test_dpclient(30, 128, 50, 1e-3)

Client-0: 13802 samples, sigma=0.48152923583984375
Client-1: 10591 samples, sigma=0.4994773864746094
Client-2: 14874 samples, sigma=0.47641754150390625
Client-3: 8293 samples, sigma=0.5178070068359375
Client-4: 12440 samples, sigma=0.4879570007324219
[1 3 0 2 4] in this round
Client-1: ε=6.924 (target=50), δ=0.001
Client-3: ε=6.829 (target=50), δ=0.001
Client-0: ε=7.026 (target=50), δ=0.001
Client-2: ε=7.081 (target=50), δ=0.001
Client-4: ε=6.988 (target=50), δ=0.001
Round  1: Training acc: 17.72%, global loss 2.239
[1 0 3 4 2] in this round
Client-1: ε=9.591 (target=50), δ=0.001
Client-0: ε=9.696 (target=50), δ=0.001
Client-3: ε=9.493 (target=50), δ=0.001
Client-4: ε=9.657 (target=50), δ=0.001
Client-2: ε=9.763 (target=50), δ=0.001
Round  2: Training acc: 41.72%, global loss 2.094
[4 2 1 0 3] in this round
Client-4: ε=11.885 (target=50), δ=0.001
Client-2: ε=12.001 (target=50), δ=0.001
Client-1: ε=11.819 (target=50), δ=0.001
Client-0: ε=11.924 (target=50), δ=0.001
Client-3: ε=11.722 (t

In [11]:
test_dpclient(30, 256, 50, 1e-3)

Client-0: 11743 samples, sigma=0.5469894409179688
Client-1: 11411 samples, sigma=0.5489730834960938
Client-2: 11778 samples, sigma=0.5450439453125
Client-3: 14212 samples, sigma=0.5298805236816406
Client-4: 10856 samples, sigma=0.5531692504882812
[0 2 4 1 3] in this round
Client-0: ε=6.692 (target=50), δ=0.001
Client-2: ε=6.701 (target=50), δ=0.001
Client-4: ε=6.666 (target=50), δ=0.001
Client-1: ε=6.684 (target=50), δ=0.001
Client-3: ε=6.770 (target=50), δ=0.001
Round  1: Training acc: 17.26%, global loss 2.279
[4 1 2 3 0] in this round
Client-4: ε=9.327 (target=50), δ=0.001
Client-1: ε=9.346 (target=50), δ=0.001
Client-2: ε=9.363 (target=50), δ=0.001
Client-3: ε=9.434 (target=50), δ=0.001
Client-0: ε=9.354 (target=50), δ=0.001
Round  2: Training acc: 20.20%, global loss 2.233
[3 1 2 0 4] in this round
Client-3: ε=11.663 (target=50), δ=0.001
Client-1: ε=11.575 (target=50), δ=0.001
Client-2: ε=11.592 (target=50), δ=0.001
Client-0: ε=11.583 (target=50), δ=0.001
Client-4: ε=11.557 (targe

In [12]:
test_dpclient(15, 128, 100, 5e-3)

Client-0: 11743 samples, sigma=0.35449981689453125
Client-1: 11411 samples, sigma=0.35521507263183594
Client-2: 11778 samples, sigma=0.3541374206542969
Client-3: 14212 samples, sigma=0.34829139709472656
Client-4: 10856 samples, sigma=0.35709381103515625
[3 1 0 4 2] in this round
Client-3: ε=16.976 (target=100), δ=0.005
Client-1: ε=16.900 (target=100), δ=0.005
Client-0: ε=16.908 (target=100), δ=0.005
Client-4: ε=16.879 (target=100), δ=0.005
Client-2: ε=16.967 (target=100), δ=0.005
Round  1: Training acc: 34.08%, global loss 2.238
[3 1 0 4 2] in this round
Client-3: ε=25.178 (target=100), δ=0.005
Client-1: ε=25.100 (target=100), δ=0.005
Client-0: ε=25.108 (target=100), δ=0.005
Client-4: ε=25.078 (target=100), δ=0.005
Client-2: ε=25.199 (target=100), δ=0.005
Round  2: Training acc: 41.21%, global loss 2.138
[3 1 2 4 0] in this round
Client-3: ε=32.325 (target=100), δ=0.005
Client-1: ε=32.248 (target=100), δ=0.005
Client-2: ε=32.376 (target=100), δ=0.005
Client-4: ε=32.226 (target=100), δ=

In [ ]:
def test_krum_dpclient(rounds, batch_size, target_epsilon, target_delta=1.25e-4):
    
    server = KrumServer(MnistCNN(num_channels, num_classes), num_attackers, device=device)
        # server = TrimmedMeanServer(CNNMnist(num_channels, num_classes), num_attackers=1, device=device)

    clients = [
            BackdoorClient(f"BClient-{i}",
                    dataset_clients[i], MnistCNN,
                    net_para = dict(num_channels=num_channels, num_classes=num_classes),
                    batch_size=batch_size, num_epochs=local_epochs,
                    trigger_fn=trigger_fn,
                    target_label=target_label,
                    device=device)
            if i < num_attackers else
            DPClient(f"Client-{i}",
                    dataset_clients[i], MnistCNN,
                    net_para = dict(num_channels=num_channels, num_classes=num_classes),
                    batch_size=batch_size, num_epochs=local_epochs,
                    total_rounds=rounds, target_epsilon=target_epsilon, target_delta=target_delta,
                    device=device)
                    for i in range(num_clients)
            ]
    
    print (f"{clients[0].name}: {clients[0].data_size} samples")
    for c in clients[1:]:
        print(f"{c.name}: {c.data_size} samples, {c.noise_info}")

    for round in range(1, rounds + 1):
        train(server, clients)

        acc_train, loss_avg = evaluate(server.net, dataset_train, device)
        acc_attacher, _ = evaluate(clients[0].net, clients[0].train_data, device)

        print(f"Round {round:2d}: Training acc: {100*acc_train:.2f}%, global loss {loss_avg:.3f}")
        print(f"Round {round:2d}: Attacker training acc: {100*acc_attacher:.2f}%")
        asr_global = evaluate_asr(server.net, dataset_test, trigger_fn, target_label, device)
        asr_local = evaluate_asr(clients[0].net, dataset_test, trigger_fn, target_label, device)
        print(f"Round {round:2d}: global ASR={100*asr_global:.2f}%, attacker-local ASR={100*asr_local:.2f}%")

    acc_test, _ = evaluate(server.net, dataset_test, device)
    print(f"Testing accuracy: {100*acc_test:.2f}%")
    
    
    for c in clients[1:]:
        c.cleanup()

test_krum_dpclient(15, 128, 100, 5e-3)

C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


BClient-0: 11743 samples
Client-1: 11411 samples, sigma=0.35521507263183594
Client-2: 11778 samples, sigma=0.3541374206542969
Client-3: 14212 samples, sigma=0.34829139709472656
Client-4: 10856 samples, sigma=0.35709381103515625
[0 4 2 1 3] in this round
BClient-0 loss in epoch 0: 2.076
BClient-0 loss in epoch 1: 1.889
BClient-0 loss in epoch 2: 1.727
BClient-0 loss in epoch 3: 1.515
BClient-0 loss in epoch 4: 1.340


c:\Users\yuloeh\Projects\Fed-Backdoor\sqrlearning\client\DPClient.py:47: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Client-4: ε=16.879 (target=100), δ=0.005
Client-2: ε=16.967 (target=100), δ=0.005
Client-1: ε=16.900 (target=100), δ=0.005
Client-3: ε=16.976 (target=100), δ=0.005
Krum: ['Client-1', 'Client-3', 'Client-4', 'Client-2', 'BClient-0'] (m=2, scores=['0.84', '0.88', '0.90', '0.92', '3.21'])
Round  1: Training acc: 16.69%, global loss 2.277
Round  1: Attacker training acc: 68.88%
Round  1: global ASR=5.85%, attacker-local ASR=55.55%
[4 2 1 3 0] in this round
Client-4: ε=25.078 (target=100), δ=0.005
Client-2: ε=25.199 (target=100), δ=0.005
Client-1: ε=25.100 (target=100), δ=0.005
Client-3: ε=25.178 (target=100), δ=0.005
BClient-0 loss in epoch 0: 2.039
BClient-0 loss in epoch 1: 1.806
BClient-0 loss in epoch 2: 1.564
BClient-0 loss in epoch 3: 1.393
BClient-0 loss in epoch 4: 1.278
Krum: ['Client-1', 'Client-3', 'Client-4', 'Client-2', 'BClient-0'] (m=2, scores=['0.77', '0.80', '0.84', '0.86', '3.30'])
Round  2: Training acc: 27.16%, global loss 2.178
Round  2: Attacker training acc: 69.22%
R

In [ ]:
from sqrlearning.client import KrumBackdoorClient, KrumProxyClient
def test_krum_dpclient_attacker(rounds, batch_size, target_epsilon, target_delta=1.25e-4):
    
    server = KrumServer(MnistCNN(num_channels, num_classes), num_attackers, device=device)
        # server = TrimmedMeanServer(CNNMnist(num_channels, num_classes), num_attackers=1, device=device)

    clients = [
            KrumBackdoorClient(f"BClient-{i}",
                    dataset_clients[i], MnistCNN,
                    net_para = dict(num_channels=num_channels, num_classes=num_classes),
                    batch_size = batch_size, num_epochs=local_epochs,
                    trigger_fn = trigger_fn,
                    target_label = target_label,
                    max_deviation_ratio=0.06,
                    device=device)
            if i < num_attackers else
            DPClient(f"Client-{i}",
                    dataset_clients[i], MnistCNN,
                    net_para = dict(num_channels=num_channels, num_classes=num_classes),
                    batch_size=batch_size, num_epochs=local_epochs,
                    total_rounds=rounds, target_epsilon=target_epsilon, target_delta=target_delta,
                    device=device)
                    for i in range(num_clients)
            ]
    
    print (f"{clients[0].name}: {clients[0].data_size} samples")
    for c in clients[1:]:
        print(f"{c.name}: {c.data_size} samples, {c.noise_info}")

    for round in range(1, rounds + 1):
        train(server, clients)

        acc_train, loss_avg = evaluate(server.net, dataset_train, device)
        acc_attacher, _ = evaluate(clients[0].net, clients[0].train_data, device)

        print(f"Round {round:2d}: Training acc: {100*acc_train:.2f}%, global loss {loss_avg:.3f}")
        print(f"Round {round:2d}: Attacker training acc: {100*acc_attacher:.2f}%")
        asr_global = evaluate_asr(server.net, dataset_test, trigger_fn, target_label, device)
        asr_local = evaluate_asr(clients[0].net, dataset_test, trigger_fn, target_label, device)
        print(f"Round {round:2d}: global ASR={100*asr_global:.2f}%, attacker-local ASR={100*asr_local:.2f}%")

    acc_test, _ = evaluate(server.net, dataset_test, device)
    print(f"Testing accuracy: {100*acc_test:.2f}%")
    
    
    for c in clients[1:]:
        c.cleanup()

test_krum_dpclient_attacker(15, 128, 100, 5e-3)

C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


BClient-0: 8095 samples
Client-1: 6587 samples, sigma=0.3744697570800781
Client-2: 14347 samples, sigma=0.34801483154296875
Client-3: 15310 samples, sigma=0.34618377685546875
Client-4: 15661 samples, sigma=0.3454303741455078
[3 0 2 4 1] in this round


c:\Users\yuloeh\Projects\Fed-Backdoor\sqrlearning\client\DPClient.py:47: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Client-3: ε=16.999 (target=100), δ=0.005
Client-2: ε=16.979 (target=100), δ=0.005
Client-4: ε=17.049 (target=100), δ=0.005
Client-1: ε=16.698 (target=100), δ=0.005
Krum: ['Client-1', 'Client-2', 'Client-3', 'Client-4', 'BClient-0'] (m=2, scores=['1.10', '1.17', '1.20', '1.40', '2.02'])
Round  1: Training acc: 11.29%, global loss 2.301
Round  1: Attacker training acc: 32.86%
Round  1: global ASR=0.00%, attacker-local ASR=0.00%
[4 0 3 1 2] in this round
Client-4: ε=25.277 (target=100), δ=0.005
Client-3: ε=25.203 (target=100), δ=0.005
Client-1: ε=24.889 (target=100), δ=0.005
Client-2: ε=25.182 (target=100), δ=0.005
Krum: ['Client-1', 'Client-2', 'Client-3', 'Client-4', 'BClient-0'] (m=2, scores=['0.97', '1.00', '1.07', '1.32', '2.22'])
Round  2: Training acc: 19.06%, global loss 2.332
Round  2: Attacker training acc: 43.69%
Round  2: global ASR=0.66%, attacker-local ASR=0.00%
[4 3 0 2 1] in this round
Client-4: ε=32.444 (target=100), δ=0.005
Client-3: ε=32.350 (target=100), δ=0.005
Client

KeyboardInterrupt: 